In [ ]:
import pandas as pd
import pulp

# 1. Inputs

# The number of minutes that a fractal center (f=4) needs to handle each week
# Sum the total minute requirements of the 20 centers of 'parts organization' and then divide by 4.
process_requirements = {
    'A': (30607.00 + 19915.0 + 13511.00 + 4377.0) / 4,
    'B': (153035.00 + 14514.5 + 27022.00 + 15735.0 + 7295.0) / 4,
    'C': (15303.50 + 7966.0 + 6220.5 + 7867.5 + 2918.0 + 15495.0) / 4,
    'D': (76517.50 + 39830.0 + 24882.0 + 40533.00 + 36715.0 + 17508.0 + 72310.0) / 4,
    'E': (20660.0 + 7553.75 + 16755.0 + 9949.50 + 29180.0 + 14340.0 + 8962.5 + 26755.00 + 2266.5) / 4,
    'F': (30990.0 + 23457.0 + 11607.75 + 14340.0 + 16132.5 + 40132.50 + 3777.5 + 22582.5) / 4,
    'G': (3377.75 + 16755.0 + 8754.0 + 7170.0 + 5377.5 + 13377.50 + 1511.0) / 4,
    'H': (15932.0 + 16888.75 + 5836.0 + 12086.00 + 14340.0 + 46821.25 + 7555.0 + 90330.0) / 4,
    'I': (38258.75 + 12441.0 + 18357.5 + 7295.0 + 25825.0 + 6043.00 + 16755.0 + 8291.25 + 8754.0 + 17925.0 + 8962.5 + 22582.5) / 4,
    'J': (76517.50 + 39830.0 + 20735.0 + 16049.0 + 41320.0 + 3021.50 + 40212.0 + 13266.00 + 17925.0 + 7555.0 + 45165.0) / 4,
    'K': (5377.5 + 21566.25 + 16293.0) / 4,
    'L': (21510.0 + 21566.25 + 124410.25 + 43448.0) / 4,
    'M': (25095.0 + 64698.75 + 182956.25 + 65172.0) / 4
}
ALL_PROCESSES = list(process_requirements.keys())

# The effective working minutes per machine per week
EFFECTIVE_MINS_PER_WEEK = 4233.60

# Equipment specifications, costs and capabilities
machine_data = {
    # A-D 
    "A": {"cost": 150000, "processes": ["A"]},
    "B": {"cost": 200000, "processes": ["B"]},
    "C": {"cost": 250000, "processes": ["C"]},
    "D": {"cost": 300000, "processes": ["D"]},
    "AB": {"cost": 250000, "processes": ["A", "B"]},
    "AC": {"cost": 280000, "processes": ["A", "C"]},
    "CD": {"cost": 350000, "processes": ["C", "D"]},
    "ABC": {"cost": 300000, "processes": ["A", "B", "C"]},
    "ABCD": {"cost": 400000, "processes": ["A", "B", "C", "D"]},
    # E-G 
    "E": {"cost": 400000, "processes": ["E"]},
    "F": {"cost": 400000, "processes": ["F"]},
    "G": {"cost": 400000, "processes": ["G"]},
    "EF": {"cost": 450000, "processes": ["E", "F"]},
    "EG": {"cost": 450000, "processes": ["E", "G"]},
    "FG": {"cost": 450000, "processes": ["F", "G"]},
    "EFG": {"cost": 500000, "processes": ["E", "F", "G"]},
    # H-J 
    "H": {"cost": 1000000, "processes": ["H"]},
    "I": {"cost": 500000, "processes": ["I"]},
    "J": {"cost": 500000, "processes": ["J"]},
    "IJ": {"cost": 750000, "processes": ["I", "J"]},
    # K-M 
    "K": {"cost": 80000, "processes": ["K"]},
    "L": {"cost": 100000, "processes": ["L"]},
    "M": {"cost": 50000, "processes": ["M"]},
    "KL": {"cost": 140000, "processes": ["K", "L"]},
    "KM": {"cost": 100000, "processes": ["K", "M"]},
    "LM": {"cost": 120000, "processes": ["L", "M"]},
    "KLM": {"cost": 150000, "processes": ["K", "L", "M"]}
}
MACHINE_NAMES = list(machine_data.keys())


# 2. initialize
prob = pulp.LpProblem("Fractal_F1_Machine_Mix", pulp.LpMinimize)

# 3. decision variable
n_machines = pulp.LpVariable.dicts(
    "N",
    MACHINE_NAMES,
    lowBound=0,
    cat='Integer'
)
time_allocations = pulp.LpVariable.dicts(
    "T",
    [(m, p) for m in MACHINE_NAMES for p in machine_data[m]["processes"]],
    lowBound=0,
    cat='Continuous'
)

# 4. obj.
prob += pulp.lpSum([
    machine_data[m]["cost"] * n_machines[m] for m in MACHINE_NAMES
]), "Total_Equipment_Cost"

# 5. Define constraints

# 约束 1: 工艺需求必须满足 (Demand Satisfaction)
for p in ALL_PROCESSES:
    prob += pulp.lpSum([
        time_allocations[(m, p)] for m in MACHINE_NAMES
        if p in machine_data[m]["processes"]
    ]) >= process_requirements[p], f"Meet_Demand_for_{p}"

# 约束 2: 设备产能不能超 (Capacity Limitation)
for m in MACHINE_NAMES:
    prob += pulp.lpSum([
        time_allocations[(m, p)] for p in machine_data[m]["processes"]
    ]) <= EFFECTIVE_MINS_PER_WEEK * n_machines[m], f"Capacity_Limit_for_{m}"

# --- 6. Solve the Model ---
solver_to_use = pulp.PULP_CBC_CMD(msg=False) # msg=False to turn off verbose logs
prob.solve(solver_to_use)

# --- 7. Print the Results ---
print(f"--- Solve Complete ---")

# We compare prob.status to pulp.LpStatusOptimal (which is 1)
print(f"Status: {pulp.LpStatus[prob.status]}")

if prob.status == pulp.LpStatusOptimal: # <-- This is the fix
    
    # Print Total Cost
    total_cost_f1 = pulp.value(prob.objective)
    print(f"\n--- Optimal Equipment Plan for 'Fractal Center F1' ---")
    print(f"Minimum Equipment Investment for a Single Center (F1): ${total_cost_f1:,.2f}")
    
    # Calculate Total Factory Investment
    total_factory_cost = total_cost_f1 * 4 # f=4
    print(f"Total Factory Equipment Investment (F1, F2, F3, F4): ${total_factory_cost:,.2f}")

    print("\n--- Optimal Machine Mix for F1 Center ---")
    total_machines_f1 = 0
    machine_mix_f1 = {}
    for m in MACHINE_NAMES:
        if n_machines[m].varValue is not None:
            count = int(round(n_machines[m].varValue))
            if count > 0:
                print(f"  {m}: {count} units")
                total_machines_f1 += count
                machine_mix_f1[m] = count
                
    print(f"  ---------------------")
    print(f"  Total Machines in F1 Center: {total_machines_f1} units")
    print(f"  Total Factory Machines (f=4): {total_machines_f1 * 4} units")

    print("\n--- Time Allocations for F1 Center (showing > 0) ---")
    time_allocations_f1 = {}
    for (m, p) in time_allocations:
        if time_allocations[(m, p)].varValue is not None and time_allocations[(m, p)].varValue > 0.1:
            alloc_time = time_allocations[(m, p)].varValue
            print(f"  {m} -> {p}: {alloc_time:,.1f} minutes")
            if (m,p) not in time_allocations_f1:
                time_allocations_f1[(m,p)] = 0
            time_allocations_f1[(m,p)] += alloc_time

else:
    print("Optimal solution not found. Please check the solver logs.")

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import io

# --- 1. 基础数据准备 ---

# (A) F1 中心的每周零件需求 (总需求 / 4)
# (数据来自您在 Task 1 中计算的 Target_Weekly_Capacity_Final)
part_demand_f1 = {
    'P1': 30607.0 / 4, 'P2': 15932.0 / 4, 'P3': 8294.0 / 4, 'P4': 13511.0 / 4,
    'P5': 10490.0 / 4, 'P6': 5836.0 / 4, 'P7': 20660.0 / 4, 'P8': 6043.0 / 4,
    'P9': 13404.0 / 4, 'P10': 6633.0 / 4, 'P11': 11672.0 / 4, 'P12': 14340.0 / 4,
    'P13': 7170.0 / 4, 'P14': 26755.0 / 4, 'P15': 3022.0 / 4, 'P16': 18066.0 / 4,
    'P17': 7170.0 / 4, 'P18': 17253.0 / 4, 'P19': 29273.0 / 4, 'P20': 21724.0 / 4
}

# (B) F1 中心的设备清单 (来自您的优化结果图片)
machine_mix_f1 = {
    'A': 4, 'B': 13, 'C': 3, 'ABCD': 19, 'EF': 15, 'FG': 7,
    'H': 13, 'I': 12, 'J': 19, 'KL': 3, 'LM': 32
}

# (C) 零件工艺路线 (来自 'Parts Specs.csv')
# 为了运行，我们直接使用 'Parts Specs.csv' 的内容
parts_specs_csv = """
Part,Process_1,Process_2,Process_3,Process_4,Process_5,Process_6,Process_7,Process_8
P1,A,B,C,D,I,J,,
P2,A,C,D,H,J,,,
P3,B,C,D,I,J,,,
P4,A,B,D,G,H,,,
P5,B,C,D,I,,,,
P6,A,B,C,D,H,I,J,
P7,C,D,E,F,I,J,,
P8,E,H,I,J,,,,
P9,E,F,G,I,J,,,
P10,E,F,I,J,,,,
P11,E,G,I,,,,,
P12,E,F,G,I,J,,,
P13,E,F,G,H,I,,,
P14,E,F,G,H,,,,
P15,E,F,G,H,J,,,
P16,F,H,I,J,,,,
P17,K,L,M,,,,,
P18,K,L,M,,,,,
P19,L,M,,,,,,
P20,K,L,M,,,,,
"""

# (D) 设备面积 (来自 'Basic Layouts.csv')
basic_layouts_csv = """
Machine,Length,Width,Area
A,15,10,150
B,15,15,225
C,15,15,225
D,30,20,600
E,20,15,300
F,20,15,300
G,20,15,300
H,40,20,800
I,20,20,400
J,20,20,400
K,10,10,100
L,10,10,100
M,10,10,100
AB,20,15,300
AC,20,15,300
CD,30,20,600
ABC,30,15,450
ABCD,30,20,600
EF,10,10,100
EG,20,15,300
FG,10,10,100
EFG,20,15,300
IJ,25,20,500
KL,10,10,100
KM,25,10,250
LM,10,10,100
KLM,10,10,100
"""

# --- 2. 定义 F1 中心的6个“布局块” ---
# 基于优化结果，我们将工艺 (A-M) 映射到它们所属的机器组
process_to_group_map = {
    'A': 'ABCD_Group', 'B': 'ABCD_Group', 'C': 'ABCD_Group', 'D': 'ABCD_Group',
    'E': 'EFG_Group', 'F': 'EFG_Group', 'G': 'EFG_Group',
    'H': 'H_Group',
    'I': 'I_Group',
    'J': 'J_Group',
    'K': 'KLM_Group', 'L': 'KLM_Group', 'M': 'KLM_Group'
}
groups = sorted(list(set(process_to_group_map.values())))
# groups = ['ABCD_Group', 'EFG_Group', 'H_Group', 'I_Group', 'J_Group', 'KLM_Group']

# --- 3. 计算每个“块”的面积 (交付成果 3 的基础) ---
print("--- 阶段 2.1: 'F1 中心' 布局块面积计算 (交付成果 3 基础) ---")
layout_df = pd.read_csv(io.StringIO(basic_layouts_csv))
layout_df = layout_df.set_index('Machine')
# 将 'Area' 转换为数值
layout_df['Area'] = pd.to_numeric(layout_df['Area'])

# 定义哪些机器属于哪个块 (基于 F1 machine_mix)
group_machines = {
    'ABCD_Group': ['A', 'B', 'C', 'ABCD'],
    'EFG_Group': ['EF', 'FG'], # E, F, G 专机未被购买
    'H_Group': ['H'],
    'I_Group': ['I'],
    'J_Group': ['J'],
    'KLM_Group': ['KL', 'LM'] # K, L, M 专机未被购买
}

block_areas = {}
for group, machines_in_group in group_machines.items():
    total_area = 0
    for machine_name, count in machine_mix_f1.items():
        if machine_name in machines_in_group:
            total_area += layout_df.loc[machine_name, 'Area'] * count
    block_areas[group] = total_area

# 打印面积结果
area_df = pd.DataFrame.from_dict(block_areas, orient='index', columns=['Total_Area_sqft'])
print(area_df.to_markdown(floatfmt=",.0f"))
print("\n")


# --- 4. 计算“块”之间的流量矩阵 (交付成果 4) ---
print("--- 阶段 2.2: 'F1 中心' 内部流量矩阵 (交付成果 4) ---")
specs_df = pd.read_csv(io.StringIO(parts_specs_csv))
specs_df = specs_df.set_index('Part')

# 初始化一个空的 From-To 矩阵
flow_matrix = pd.DataFrame(0, index=groups, columns=groups, dtype=float)

# 遍历所有20个零件
for part, routing in specs_df.iterrows():
    demand = part_demand_f1.get(part, 0)
    if demand == 0:
        continue
        
    # 获取工艺步骤列表，并去除空值
    steps = [step for step in routing if pd.notna(step)]
    
    # 遍历工艺路线 (例如 A -> B, B -> C, ...)
    for i in range(len(steps) - 1):
        process_from = steps[i]
        process_to = steps[i+1]
        
        # 映射到对应的“块”
        group_from = process_to_group_map[process_from]
        group_to = process_to_group_map[process_to]
        
        # 仅当流量在 *不同* 块之间移动时，我们才记录它
        # (块内部的流量对于块布局不重要)
        if group_from != group_to:
            flow_matrix.loc[group_from, group_to] += demand

# 打印流量矩阵结果
print("流量单位: 周转次数/周 (Trips/Week)")
# 四舍五入到整数，便于阅读
flow_matrix = flow_matrix.round(0).astype(int)
print(flow_matrix.to_markdown())

--- 阶段 2.1: 'F1 中心' 布局块面积计算 (交付成果 3 基础) ---
|            |   Total_Area_sqft |
|:-----------|------------------:|
| ABCD_Group |            15,600 |
| EFG_Group  |             2,200 |
| H_Group    |            10,400 |
| I_Group    |             4,800 |
| J_Group    |             7,600 |
| KLM_Group  |             3,500 |


--- 阶段 2.2: 'F1 中心' 内部流量矩阵 (交付成果 4) ---
流量单位: 周转次数/周 (Trips/Week)
|            |   ABCD_Group |   EFG_Group |   H_Group |   I_Group |   J_Group |   KLM_Group |
|:-----------|-------------:|------------:|----------:|----------:|----------:|------------:|
| ABCD_Group |            0 |        8543 |      5442 |     12348 |         0 |           0 |
| EFG_Group  |            0 |           0 |     18642 |     16677 |         0 |           0 |
| H_Group    |            0 |           0 |         0 |      9279 |      4738 |           0 |
| I_Group    |            0 |           0 |         0 |         0 |     30971 |           0 |
| J_Group    |            0 |           0 | 

C:\Users\11573\AppData\Local\Temp\ipykernel_26056\2022064120.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7651.75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  flow_matrix.loc[group_from, group_to] += demand
C:\Users\11573\AppData\Local\Temp\ipykernel_26056\2022064120.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7651.75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  flow_matrix.loc[group_from, group_to] += demand
C:\Users\11573\AppData\Local\Temp\ipykernel_26056\2022064120.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3377.75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  flow_matrix.loc[